<a href="https://colab.research.google.com/github/talpt/pyton/blob/main/Ema_Dizilimi_Basit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================================================================================
#
#                           ÇOK STRATEJİLİ EMA & TEKNİK ANALİZ TARAMA ROBOTU (v3.1)
#
# ======================================================================================================================
#
# --- AÇIKLAMA ---
# Bu script, Borsa İstanbul (BIST) hisselerini üç farklı güçlü EMA stratejisine göre tarar:
#
#   1. EMA Destek Stratejisi:
#      Büyük bir EMA'yı (örn: 89) destek yapmış ve tepki verme potansiyeli olan hisseleri bulur.
#
#   2. İdeal EMA Dizilim Stratejisi:
#      EMA'ların (13>21>34) mükemmel bir boğa trendi sıralamasına geçtiği, momentumu güçlü hisseleri tespit eder.
#
#   3. EMA Sıkışma Stratejisi (GÜNCELLENDİ):
#      5 ana EMA'nın (13, 21, 34, 55, 89) birbirine çok yaklaştığı, bir "yumak" haline geldiği hisseleri
#      tespit eder. Bu durum, genellikle ardından gelecek sert bir fiyat hareketinin habercisidir.
#      (Not: Haftalık periyotta daha çok hisse bulunabilmesi için EMA144 bu moddan çıkarılmıştır.)
#
# --- YASAL UYARI ---
# Bu script, yatırım tavsiyesi niteliği taşımaz. Eğitim ve teknik analiz pratikleri amacıyla geliştirilmiştir.
#
# ======================================================================================================================

# ✅ GEREKLİ KURULUMLAR
!pip install git+https://github.com/rongardF/tvdatafeed -q
!pip install tradingview-screener==2.5.0 -q
!pip install pandas numpy tqdm openpyxl -q

# Kütüphaneleri içeri aktarır.
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from datetime import datetime
from IPython.display import display, clear_output
from tradingview_screener import get_all_symbols
from tvDatafeed import TvDatafeed, Interval
import warnings

# UYARI MESAJLARINI GİZLEME (Temiz Çıktı İçin)
warnings.filterwarnings('ignore', category=FutureWarning)

# Pandas görüntüleme ayarları.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.colheader_justify', 'left')

# --- Tarama sonuç DataFrame'ini görsel olarak biçimlendiren fonksiyon. ---
def format_tarama_sonucu(df):
    """DataFrame'i profesyonel ve okunabilir bir tabloya dönüştürür."""
    # Tablo stilini ayarla - SOLA DAYALI
    style_df = df.style.set_properties(**{
        'text-align': 'left',
        'border': '1px solid #ddd',
        'padding': '8px'
    })

    # Başlık ve hücre stillerini ayarla
    style_df = style_df.set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('text-align', 'left'),
                ('background-color', '#2A5CAA'),
                ('color', 'white'),
                ('font-weight', 'bold'),
                ('padding', '10px'),
                ('border', '1px solid #1E3A5F')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('text-align', 'left'),
                ('padding', '8px'),
                ('border', '1px solid #ddd')
            ]
        },
        {
            'selector': 'table',
            'props': [
                ('border-collapse', 'collapse'),
                ('width', '100%'),
                ('margin', '20px 0')
            ]
        }
    ])

    # Sıkışma %
    if 'Sıkışma %' in df.columns:
        style_df = style_df.background_gradient(subset=['Sıkışma %'], cmap='Greens_r')

    # Son Mum
    if 'Son Mum' in df.columns:
        style_df = style_df.map(lambda x: 'color: green; font-weight: bold;' if x == 'Yeşil' else 'color: red;', subset=['Son Mum'])

    # Hacim Atak (10 Mum)
    if 'Hacim Atak (10 Mum)' in df.columns:
        style_df = style_df.background_gradient(subset=['Hacim Atak (10 Mum)'], cmap='Greens')

    # Uzaklık Seviyesi
#    if 'Uzaklık Seviyesi' in df.columns:
#        def uzaklik_renklendir(val):
#            if val == 'Düşük': return 'background-color: #2ECC71; color: white; font-weight: bold;'
#            elif val == 'Orta': return 'background-color: #F39C12; color: white; font-weight: bold;'
#            elif val == 'Yüksek': return 'background-color: #E74C3C; color: white; font-weight: bold;'
#            return ''
#        style_df = style_df.map(uzaklik_renklendir, subset=['Uzaklık Seviyesi'])

    # Trend Gücü
    if 'Trend Gücü' in df.columns:
        def trend_renklendir(val):
            if '🔥' in str(val) or '💪' in str(val): return 'color: #27AE60; font-weight: bold;'
            elif '📈' in str(val) or '🌱' in str(val): return 'color: #3498DB; font-weight: bold;'
            elif '⚠️' in str(val): return 'color: #E67E22;'
            elif '📉' in str(val): return 'color: #E74C3C;'
            else: return ''
        style_df = style_df.map(trend_renklendir, subset=['Trend Gücü'])

    # EMA Sıralaması (Dinamik sütun adı)
    ema_siralama_cols = [col for col in df.columns if 'EMA Sıralaması' in col]
    if ema_siralama_cols:
        style_df = style_df.map(lambda val: 'color: green; font-weight: bold;' if val == '✅' else 'color: red;', subset=ema_siralama_cols)

    # Direnç Durumu
    if 'Direnç Durumu' in df.columns:
        style_df = style_df.map(lambda val: 'color: #27AE60; font-weight: bold;' if val == 'Önü Açık' else 'color: #E74C3C;', subset=['Direnç Durumu'])

    # 🆕 YENİ SÜTUNLAR
#    if 'Son 10 Yeşil Mum' in df.columns:
#        style_df = style_df.background_gradient(subset=['Son 10 Yeşil Mum'], cmap='Greens', vmin=0, vmax=10)

#    if 'Yeşil Hacim Atak' in df.columns:
#        style_df = style_df.background_gradient(subset=['Yeşil Hacim Atak'], cmap='Greens', vmin=0, vmax=10)

    if 'Merdiven Hacim' in df.columns:
        style_df = style_df.map(lambda val: 'color: green; font-weight: bold;' if '✅' in str(val) else 'color: red;', subset=['Merdiven Hacim'])

    if 'Hacim Gücü %' in df.columns:
        style_df = style_df.background_gradient(subset=['Hacim Gücü %'], cmap='Blues', vmin=50, vmax=150)

    # 🆕 YENİ SÜTUNLAR İÇİN STİL
    if 'EMA Direnci' in df.columns:
        style_df = style_df.map(
            lambda val: 'color: #27AE60; font-weight: bold;' if val == 'Önü Açık'
            else 'color: #E67E22;' if 'Veri Yetersiz' in str(val) or 'Kısmi Veri' in str(val)
            else 'color: #E74C3C;',
            subset=['EMA Direnci']
        )

    if 'Sol Tepe Durumu' in df.columns:
        def sol_tepe_renklendir(val):
            if 'Testinde' in str(val):
                return 'color: #F39C12; font-weight: bold;'  # Turuncu
            elif 'Direnci' in str(val):
                return 'color: #E74C3C; font-weight: bold;'  # Kırmızı
            elif 'Uzak' in str(val):
                return 'color: #95A5A6;'  # Gri
            elif 'Geçildi' in str(val):
                return 'color: #27AE60; font-weight: bold;'  # Yeşil
            else:
                return ''
        style_df = style_df.map(sol_tepe_renklendir, subset=['Sol Tepe Durumu'])

    if 'Direnç Mesafesi %' in df.columns:
        style_df = style_df.background_gradient(subset=['Direnç Mesafesi %'], cmap='RdYlGn_r', vmin=-10, vmax=0)

    # 🆕 VERİ KADEMESİ RENKLENDIRME
#    if 'Veri Kademesi' in df.columns:
#        def kademe_renklendir(val):
#            if val == 'TAM':
#                return 'background-color: #27AE60; color: white; font-weight: bold;'
#            elif val == 'ORTA':
#                return 'background-color: #F39C12; color: white; font-weight: bold;'
#            elif val == 'MİNİMUM':
#                return 'background-color: #E67E22; color: white; font-weight: bold;'
#            return ''
#        style_df = style_df.map(kademe_renklendir, subset=['Veri Kademesi'])

    # Format
    format_dict = {
        'Fiyat-Ref. Uzaklık %': '{:.2f}%',
        'Sıkışma %': '{:.2f}%',
        'Hacim Gücü %': '{:.0f}%',
        'Direnç Mesafesi %': '{:.2f}%'
    }

    style_df = style_df.format(format_dict)
    return style_df

# ============================================================================
# 🎨 PROFESYONEL KULLANICI ARAYÜZÜ
# ============================================================================
# Terminal genişliğini al
try:
    terminal_width = os.get_terminal_size().columns
except:
    terminal_width = 80

def print_header(title, emoji=""):
    """Başlık yazdırma"""
    line = "=" * terminal_width
    print(f"\n{line}")
    print(f"{emoji} {title}".center(terminal_width))
    print(f"{line}")

def print_section(title):
    """Alt başlık yazdırma"""
    line = "─" * terminal_width
    print(f"\n{line}")
    print(f"📊 {title}")
    print(f"{line}")

# --- Kullanıcıdan Parametre Alma ---
print_header("ÇOK STRATEJİLİ EMA TARAMA ROBOTU v3.2", "🚀")
print("\n💡 Bu araç, BIST hisselerini güçlü EMA stratejilerine göre tarar.")
print("⚠️  Yatırım tavsiyesi değildir. Eğitim amaçlıdır.\n")

# ============================================================================
# 📊 KOMPOZİT HESAPLAMA SEÇİMİ
# ============================================================================
print_header("📊 HİSSE FİLTRELEME SEÇİMİ")
print("""
📈 TARAMA TÜRLERİ:

1️⃣  : Tüm BIST Hisseleri (Normal Fiyat)
    └─ Hisse fiyatları olduğu gibi kullanılır
    └─ Klasik teknik analiz
    └─ Tüm stratejiler için uygun

2️⃣  : Kompozit BIST Hisseleri (Hisse / XU100)
    └─ Hisse fiyatı XU100 endeksine bölünür
    └─ Göreceli güç analizi
    └─ Endeksten bağımsız performans ölçümü

💡 ÖNERİ: Kompozit mod, hissenin piyasaya göre gerçek gücünü gösterir
""")

while True:
    kompozit_secimi = input("👉 Tarama türü seçin (1-2): ").strip()
    if kompozit_secimi == "1":
        kompozit_modu = False
        print("✅ Seçilen: Normal BIST Taraması\n")
        break
    elif kompozit_secimi == "2":
        kompozit_modu = True
        print("✅ Seçilen: Kompozit BIST Taraması (Hisse/XU100)\n")
        break
    else:
        print("❌ Geçersiz seçim! Lütfen 1 veya 2 girin.")

# ============================================================================
# 📍 STRATEJİ SEÇİMİ BÖLÜMÜ
# ============================================================================

print_header("🎯 STRATEJİ SEÇİMİ")
print("""
📈 MEVCUT STRATEJİLER:

1️⃣  : EMA Destek Stratejisi
    └─ Büyük bir EMA'dan (89, 144, 233) destek alanlar
    └─ Risk/Ödül oranı yüksek
    └─ Orta-Uzun vade için ideal

2️⃣  : İdeal EMA Dizilim Stratejisi
    └─ EMA1 > EMA2 > EMA3 sıralaması (Özelleştirilebilir!)
    └─ Güçlü momentum sinyali
    └─ Kısa-Orta vade için uygun

3️⃣  : EMA Sıkışma Stratejisi
    └─ 3a: 3 EMA Sıkışması (Hızlı sinyaller)
    └─ 3b: 4 EMA Sıkışması (Dengeli)
    └─ 3c: 5 EMA Sıkışması (Güçlü sinyaller)
    └─ Periyotlar özelleştirilebilir!
    └─ Patlama potansiyeli yüksek

💡 ÖNERİ: Yeni başlayanlar için Strateji 2 önerilir
""")

while True:
    tarama_secimi = input("👉 Strateji seçin (1/2/3a/3b/3c): ").strip()

    if tarama_secimi in ['1', '2', '3a', '3b', '3c']:
        print(f"✅ Seçilen: Strateji {tarama_secimi}\n")
        break
    else:
        print("❌ Geçersiz seçim! Lütfen 1, 2, 3a, 3b veya 3c girin.\n")

# Geçersiz seçim kontrolü
if tarama_secimi not in ['1', '2', '3a', '3b', '3c']:
    print("❌ Geçersiz seçim! Lütfen 1, 2, 3a, 3b veya 3c girin.")
    exit()

# ============================================================================
# ⏰ ZAMAN ARALIĞI SEÇİMİ
# ============================================================================
print_header("⏰ AKILLI ZAMAN ARALIĞI SEÇİMİ")

# Seans kontrolü
current_hour = datetime.now().hour
is_seans = (9 <= current_hour < 18)

if is_seans:
    print("🕐 Mevcut Durum: ✅ Seans içi - İşlemler aktif")
    print("💡 ÖNERİ: Kısa vadeli zaman dilimleri güvenilir")
else:
    print("🕐 Mevcut Durum: 🌙 Seans dışı - İşlem yok")
    print("⚠️  DİKKAT: Kısa vadeli zaman dilimleri yanıltıcı olabilir")

print(f"\n{'─' * terminal_width}")
print("""
📊 KISA VADELİ (Aktif takip gerektirir) ⚡
    └─ 1️⃣  : 1 Dakika
    └─ 2️⃣  : 5 Dakika
    └─ 3️⃣  : 15 Dakika
    └─ 4️⃣  : 30 Dakika

📈 ORTA VADELİ (Dengeli yaklaşım) ✅
    └─ 5️⃣  : 1 Saatlik
    └─ 6️⃣  : 4 Saatlik

📉 UZUN VADELİ (Güvenilir sinyaller) 🎯
    └─ 7️⃣  : Günlük
    └─ 8️⃣  : Haftalık
    └─ 9️⃣  : Aylık
""")

if not is_seans:
    print("💡 ÖNERİ: Seans dışında 6+ numaralı seçenekler önerilir\n")

interval_mapping = {
    "1": (Interval.in_1_minute, "1dk"),
    "2": (Interval.in_5_minute, "5dk"),
    "3": (Interval.in_15_minute, "15dk"),
    "4": (Interval.in_30_minute, "30dk"),
    "5": (Interval.in_1_hour, "1s"),
    "6": (Interval.in_4_hour, "4s"),
    "7": (Interval.in_daily, "Günlük"),
    "8": (Interval.in_weekly, "Haftalık"),
    "9": (Interval.in_monthly, "Aylık")
}

choice = input("👉 Zaman aralığı seçin (1-9): ").strip()
interval, interval_str = interval_mapping.get(choice, (Interval.in_daily, "Günlük"))
print(f"✅ Seçilen: {interval_str}\n")

ref_ema_period = 0
maks_sikisma_orani = 0
# ✅ YENİ DEĞİŞKENLER (İdeal Dizilim İçin)
ema1_period = 0
ema2_period = 0
ema3_period = 0

# ============================================================================
# 🔧 STRATEJİYE ÖZEL PARAMETRELER
# ============================================================================
if tarama_secimi == '1':
    print_header("📈 EMA DESTEK STRATEJİSİ AYARLARI")
    print("""
🎯 Referans EMA Önerileri:
    • EMA 89  : Orta vadeli güçlü destek
    • EMA 144 : Uzun vadeli ana destek
    • EMA 233 : Çok uzun vadeli kritik seviye

💡 Küçük periyotlar = Daha fazla sinyal (daha riskli)
💡 Büyük periyotlar = Daha az sinyal (daha güvenilir)
""")
    ref_ema_period = int(input("👉 Referans EMA periyodu (Örn: 89, 144, 233): ").strip())
    print(f"✅ Kullanılacak: EMA {ref_ema_period}\n")

elif tarama_secimi == '2':
    print_header("🎯 İDEAL EMA DİZİLİM STRATEJİSİ AYARLARI")
    print("""
📊 Popüler EMA Kombinasyonları:
    • 5-8-13    : Selçuk Gönençler
    • 8-13-21   : Kısa vadeli agresif
    • 5-14-34   : Kompozit Haftalık
    • 13-21-34  : Klasik Fibonacci (Önerilen!)
    • 21-34-55  : Orta vadeli dengeli(Haftalık 5-14-34 Kompozit Grafiklere bakarken, Normal Grafik Ekranında da 21-34 Kesişimi, Sıkışma, Yukarı Kesme takip edilebilir.)
    • 34-55-89  : Uzun vadeli konservatif

⚠️  KURAL: EMA1 < EMA2 < EMA3 olmalı (Örn: 13 < 21 < 34)
""")
    ema1_period = int(input("👉 EMA1 Periyodu (En küçük): ").strip())
    ema2_period = int(input("👉 EMA2 Periyodu (Orta): ").strip())
    ema3_period = int(input("👉 EMA3 Periyodu (En büyük): ").strip())

    if not (ema1_period < ema2_period < ema3_period):
        print("\n❌ HATA: EMA periyotları küçükten büyüğe sıralı olmalı!")
        print(f"   Girilen: {ema1_period}, {ema2_period}, {ema3_period}")
        exit()

    print(f"✅ Kullanılacak: EMA {ema1_period} > {ema2_period} > {ema3_period}\n")

elif tarama_secimi in ['3a', '3b', '3c']:
    print_header("🔥 EMA SIKIŞMA STRATEJİSİ AYARLARI")

    if tarama_secimi == '3a':
        print("\n📊 3 EMA Sıkışması Seçildi")
        print("   └─ Hızlı sinyaller, daha sık sıkışma")
        print("   └─ Örnek: 8, 13, 21")
        ema_sayisi = 3
    elif tarama_secimi == '3b':
        print("\n📊 4 EMA Sıkışması Seçildi")
        print("   └─ Dengeli, orta güvenilirlik")
        print("   └─ Örnek: 13, 21, 34, 55")
        ema_sayisi = 4
    else:  # 3c
        print("\n📊 5 EMA Sıkışması Seçildi")
        print("   └─ Güçlü sinyaller, daha az sıkışma")
        print("   └─ Örnek: 13, 21, 34, 55, 89")
        ema_sayisi = 5

    print(f"\n🔢 {ema_sayisi} adet EMA periyodu giriniz (küçükten büyüğe):")
    sikisma_ema_periyotlari = []
    for i in range(ema_sayisi):
        while True:
            try:
                period = int(input(f"   EMA {i+1}: "))
                if period > 0:
                    sikisma_ema_periyotlari.append(period)
                    break
                else:
                    print("   ⚠️  Lütfen pozitif bir sayı girin!")
            except ValueError:
                print("   ⚠️  Lütfen geçerli bir sayı girin!")

    sikisma_ema_periyotlari.sort()  # Küçükten büyüğe sırala
    print(f"\n✅ Seçilen EMA'lar: {sikisma_ema_periyotlari}")

    # Sıkışma eşiği
    oneri_esik = 2 if ema_sayisi == 3 else 3 if ema_sayisi == 4 else 5
    while True:
        try:
            sikisma_esigi = float(input(f"\n📊 Sıkışma eşiği (%) [Önerilen: {oneri_esik}]: "))
            if 0 < sikisma_esigi <= 10:
                maks_sikisma_orani = sikisma_esigi  # Eski değişkeni güncelle
                break
            else:
                print("   ⚠️  Lütfen 0-10 arası bir değer girin!")
        except ValueError:
            print("   ⚠️  Lütfen geçerli bir sayı girin!")

    print(f"✅ Kullanılacak: %{maks_sikisma_orani} ve altı\n")

else:
    print("\n❌ Geçersiz seçim! Program sonlandırılıyor...\n")
    exit()

hacim_ortalama_periyodu = 20

# ============================================================================
# 🚀 TARAMA BAŞLIYOR
# ============================================================================
print_header("🚀 TARAMA BAŞLIYOR", "⏳")

# Strateji adını belirle
if tarama_secimi == '1':
    strateji_adi = "EMA Destek"
elif tarama_secimi == '2':
    strateji_adi = "İdeal Dizilim"
elif tarama_secimi in ['3a', '3b', '3c']:
    strateji_adi = f"EMA Sıkışma ({ema_sayisi} EMA)"
else:
    strateji_adi = "Bilinmeyen"

print(f"📊 Strateji: {strateji_adi}")
print(f"⏰ Zaman Aralığı: {interval_str}")

if tarama_secimi == '1':
    print(f"📈 Referans EMA: {ref_ema_period}")
elif tarama_secimi == '2':
    print(f"🎯 EMA Dizilimi: {ema1_period} > {ema2_period} > {ema3_period}")
elif tarama_secimi in ['3a', '3b', '3c']:
    print(f"🔥 Seçilen EMA'lar: {sikisma_ema_periyotlari}")
    print(f"🔥 Sıkışma Oranı: ≤ %{maks_sikisma_orani}")

# DEĞİŞİKLİK 2: Hesaplanacak EMA listesi...

# DEĞİŞİKLİK 2: Hesaplanacak EMA listesi ve gerekli mum sayısı, seçilen stratejiye göre dinamik olarak belirleniyor.
# Eğer sıkışma modu seçildiyse, 144 periyodu hiç dahil edilmiyor.
base_ema_periods = [13, 21, 34, 55, 89]
ema_periods_to_calculate = set(base_ema_periods)

if tarama_secimi == '1':
    ema_periods_to_calculate.add(ref_ema_period)
elif tarama_secimi == '2':
    # ✅ YENİ: İdeal dizilim için kullanıcının girdiği EMA'ları ekle
    ema_periods_to_calculate.add(ema1_period)
    ema_periods_to_calculate.add(ema2_period)
    ema_periods_to_calculate.add(ema3_period)

if tarama_secimi != '3': # Sıkışma modu hariç diğer modlar için 144 EMA'yı direnç kontrolü için ekle
    ema_periods_to_calculate.add(144)

ema_periods_to_calculate = sorted(list(ema_periods_to_calculate))
if 0 in ema_periods_to_calculate: ema_periods_to_calculate.remove(0)

# ============================================================================
# 📊 KADEMELI MUM KONTROLÜ SİSTEMİ
# ============================================================================
# KADEME 1: Minimum (Taramaya dahil edilir)
minimum_ema_period = 55
gerekli_minimum_mum = minimum_ema_period + 20  # 75 mum

# KADEME 2: Orta (EMA89 + Direnç kontrolü)
orta_ema_period = 89
gerekli_orta_mum = orta_ema_period + 20  # 109 mum

# KADEME 3: Tam (EMA144 + Tam direnç kontrolü)
tam_ema_period = 144
gerekli_tam_mum = tam_ema_period + 20  # 164 mum

# Bilgilendirme
print(f"\n📊 KADEMELI MUM KONTROLÜ:")
print(f"   ├─ Minimum (Taramaya Dahil): {gerekli_minimum_mum} mum (EMA55'e kadar)")
print(f"   ├─ Orta (EMA89 + Direnç): {gerekli_orta_mum} mum")
print(f"   └─ Tam (EMA144 + Tam Direnç): {gerekli_tam_mum} mum")
print(f"\nℹ️ Hesaplanacak EMA'lar: {', '.join([f'EMA{p}' for p in ema_periods_to_calculate])}")

# ============================================================================
# 📊 KOMPOZİT MOD İÇİN XU100 VERİSİNİ ÇEK
# ============================================================================
endeks_df = None
xu100_son_fiyat = None

if kompozit_modu:
    print_header("📊 KOMPOZİT HESAPLAMA HAZIRLANIYOR")
    print("📈 BIST 100 (XU100) referans verisi çekiliyor...")

    try:
        tv = TvDatafeed()
        endeks_df = tv.get_hist('XU100', exchange='BIST', interval=interval, n_bars=gerekli_minimum_mum + 50)

        if endeks_df is None or endeks_df.empty:
            print("❌ KRİTİK HATA: XU100 endeks verisi çekilemedi!")
            print("⚠️  Kompozit hesaplama yapılamayacak. Program durduruluyor.")
            exit()
        else:
            xu100_son_fiyat = endeks_df['close'].iloc[-1]
            print(f"✅ XU100 verisi başarıyla çekildi")
            print(f"   📊 Veri Sayısı: {len(endeks_df)} mum")
            print(f"   💰 Son XU100 Değeri: {xu100_son_fiyat:,.2f}\n")

    except Exception as e:
        print(f"❌ KRİTİK HATA: XU100 verisi çekilirken hata oluştu: {e}")
        print("⚠️  Program durduruluyor.")
        exit()
else:
    print("ℹ️ Normal mod seçildi, XU100 verisi çekilmeyecek.\n")

# --- TvDatafeed ve Sembolleri Çekme ---
tv = TvDatafeed()
print("\n📈 BIST Sembolleri Çekiliyor...")
try:
    hisseler = [s.replace("BIST:", "") for s in get_all_symbols("turkey")]
    print(f"✅ {len(hisseler)} adet BIST sembolü çekildi.")
except Exception as e:
    print(f"❌ Semboller çekilirken hata oluştu: {e}")
    hisseler = []

sonuclar = []
yetersiz = []

# --- Tarama Döngüsü ---
if hisseler:
    for hisse in tqdm(hisseler, desc="📊 Hisseler Taranıyor"):
        try:
            # Maksimum mum çek (en fazla ihtiyaç duyulabilecek)
            df = tv.get_hist(hisse, exchange="BIST", interval=interval, n_bars=gerekli_tam_mum + 50)

            # Minimum kontrol (KADEME 1)
            if df is None or df.empty or len(df) < gerekli_minimum_mum:
                yetersiz.append(hisse); continue

            # Mum sayısına göre kademe belirleme
            mum_sayisi = len(df)
            if mum_sayisi >= gerekli_tam_mum:
                veri_kademesi = "TAM"
            elif mum_sayisi >= gerekli_orta_mum:
                veri_kademesi = "ORTA"
            else:
                veri_kademesi = "MİNİMUM"

            df = df.dropna(subset=["close", "open", "high", "low", "volume"]).copy()
            # ============================================================================
            # 🔥 KOMPOZİT HESAPLAMA (Hisse / XU100)
            # ============================================================================
            if kompozit_modu and endeks_df is not None:
                try:
                    # Hisse ve endeks verilerini hizala
                    hisse_kapanis, endeks_kapanis = df['close'].align(endeks_df['close'], join='inner')

                    if hisse_kapanis.empty or len(hisse_kapanis) < gerekli_minimum_mum:
                        yetersiz.append(f"{hisse} (Kompozit Hizalama Sorunu)")
                        continue

                    # Göreceli güç hesapla (Hisse / XU100 * 100)
                    goreceli_guc = (hisse_kapanis / endeks_kapanis) * 100

                    # DataFrame'i hizalanmış veriye göre güncelle
                    df = df.loc[goreceli_guc.index].copy()

                    # Tüm fiyat verilerini kompozite çevir
                    df['open'] = (df['open'] / endeks_kapanis) * 100
                    df['high'] = (df['high'] / endeks_kapanis) * 100
                    df['low'] = (df['low'] / endeks_kapanis) * 100
                    df['close'] = goreceli_guc

                except Exception as e:
                    yetersiz.append(f"{hisse} (Kompozit Hesaplama Hatası)")
                    continue

            # ============================================================================
            # 📊 KADEMELI EMA HESAPLAMA
            # ============================================================================
            # KADEME 1: Her zaman hesaplanan EMA'lar
            temel_emalar = [13, 21, 34, 55]
            for period in temel_emalar:
                if period in ema_periods_to_calculate:
                    df[f"EMA{period}"] = df["close"].ewm(span=period, adjust=False).mean()

            # Kullanıcının seçtiği EMA'ları ekle (Strateji 1 ve 2 için)
            if tarama_secimi == '1' and ref_ema_period <= 55:
                df[f"EMA{ref_ema_period}"] = df["close"].ewm(span=ref_ema_period, adjust=False).mean()
            elif tarama_secimi == '2':
                for p in [ema1_period, ema2_period, ema3_period]:
                    if p <= 55 and f"EMA{p}" not in df.columns:
                        df[f"EMA{p}"] = df["close"].ewm(span=p, adjust=False).mean()
            elif tarama_secimi in ['3a', '3b', '3c']:
                # Strateji 3 için dinamik EMA'lar (KADEME 1)
                for period in sikisma_ema_periyotlari:
                    if period <= 55 and f"EMA{period}" not in df.columns:
                        df[f"EMA{period}"] = df["close"].ewm(span=period, adjust=False).mean()


            # KADEME 2: EMA89 (Orta kademe)
            ema89_hesaplandi = False
            if veri_kademesi in ["ORTA", "TAM"]:
                df["EMA89"] = df["close"].ewm(span=89, adjust=False).mean()
                ema89_hesaplandi = True


                # Strateji 1 ve 2 için büyük EMA'lar
                if tarama_secimi == '1' and 55 < ref_ema_period <= 89:
                    df[f"EMA{ref_ema_period}"] = df["close"].ewm(span=ref_ema_period, adjust=False).mean()
                elif tarama_secimi == '2':
                    for p in [ema1_period, ema2_period, ema3_period]:
                        if 55 < p <= 89 and f"EMA{p}" not in df.columns:
                            df[f"EMA{p}"] = df["close"].ewm(span=p, adjust=False).mean()
                elif tarama_secimi in ['3a', '3b', '3c']:
                    # Strateji 3 için dinamik EMA'lar (KADEME 2)
                    for period in sikisma_ema_periyotlari:
                        if 55 < period <= 89 and f"EMA{period}" not in df.columns:
                            df[f"EMA{period}"] = df["close"].ewm(span=period, adjust=False).mean()



            # KADEME 3: EMA144 (Tam kademe)
            ema144_hesaplandi = False
            if veri_kademesi == "TAM":
                df["EMA144"] = df["close"].ewm(span=144, adjust=False).mean()
                ema144_hesaplandi = True


                # Strateji 1 için çok büyük EMA'lar
                if tarama_secimi == '1' and ref_ema_period > 89:
                    df[f"EMA{ref_ema_period}"] = df["close"].ewm(span=ref_ema_period, adjust=False).mean()
                elif tarama_secimi == '2':
                    for p in [ema1_period, ema2_period, ema3_period]:
                        if p > 89 and f"EMA{p}" not in df.columns:
                            df[f"EMA{p}"] = df["close"].ewm(span=p, adjust=False).mean()
                elif tarama_secimi in ['3a', '3b', '3c']:
                    # Strateji 3 için dinamik EMA'lar (KADEME 3)
                    for period in sikisma_ema_periyotlari:
                        if period > 89 and f"EMA{period}" not in df.columns:
                            df[f"EMA{period}"] = df["close"].ewm(span=period, adjust=False).mean()


            df['volume_ma'] = df['volume'].rolling(window=hacim_ortalama_periyodu).mean()

            df = df.dropna().copy()
            if len(df) < gerekli_minimum_mum:
                 yetersiz.append(hisse); continue

            kriter_saglandi = False
            son_fiyat = df["close"].iloc[-1]

            if tarama_secimi == '1':
                df['Ref_EMA'] = df[f'EMA{ref_ema_period}']
                kosul_kapanis_ema_ustu = son_fiyat > df["Ref_EMA"].iloc[-1]
                ema_test_edildi = any(df["low"].iloc[-i] <= df["Ref_EMA"].iloc[-i] and df["close"].iloc[-i] > df["Ref_EMA"].iloc[-i] for i in range(2, 5))
                if kosul_kapanis_ema_ustu and ema_test_edildi: kriter_saglandi = True

            elif tarama_secimi == '2':
                # ✅ YENİ: Dinamik EMA periyotları ile koşul kontrolü
                ema1_value = df[f'EMA{ema1_period}'].iloc[-1]
                ema2_value = df[f'EMA{ema2_period}'].iloc[-1]
                ema3_value = df[f'EMA{ema3_period}'].iloc[-1]

                kosul_ideal_siralama = ema1_value > ema2_value > ema3_value
                kosul_fiyat_yakin = son_fiyat > ema1_value
                if kosul_ideal_siralama and kosul_fiyat_yakin: kriter_saglandi = True



            elif tarama_secimi in ['3a', '3b', '3c']:
                # Dinamik EMA'ları al
                sıkışma_emaları = []
                for period in sikisma_ema_periyotlari:
                    if f'EMA{period}' in df.columns:
                        sıkışma_emaları.append(df[f'EMA{period}'].iloc[-1])

                if len(sıkışma_emaları) >= ema_sayisi:
                    min_ema = min(sıkışma_emaları)
                    max_ema = max(sıkışma_emaları)
                    if min_ema > 0:
                        gercek_sikisma_orani = ((max_ema - min_ema) / min_ema) * 100
                        fiyat_teyidi = son_fiyat > min_ema
                        if gercek_sikisma_orani <= maks_sikisma_orani and fiyat_teyidi:
                            kriter_saglandi = True



            if kriter_saglandi:
                son_mum_rengi = "Yeşil" if df["close"].iloc[-1] > df["open"].iloc[-1] else "Kırmızı"
                hacim_atak_sayisi = sum(1 for i in range(1, 11) if df["close"].iloc[-i] > df["open"].iloc[-i] and df["volume"].iloc[-i] > df['volume_ma'].iloc[-i])

                # ============================================================================
                # 🆕 YENİ HACİM VE TREND ANALİZLERİ
                # ============================================================================

                # 1. Son 10 mumda yeşil mum sayısı
                son_10_yesil = sum(1 for i in range(1, 11) if df["close"].iloc[-i] > df["open"].iloc[-i])

                # 2. Son 10 mumda hacim artışlı yeşil mum sayısı
                yesil_hacim_atak = sum(1 for i in range(1, 11)
                                       if df["close"].iloc[-i] > df["open"].iloc[-i]
                                       and df["volume"].iloc[-i] > df['volume_ma'].iloc[-i])

                # 3. Merdiven hacim kontrolü (Ardışık 3+ yeşil + hacim artan)
                merdiven_hacim = False
                for i in range(1, 8):
                    try:
                        if all(df["close"].iloc[-(i+j)] > df["open"].iloc[-(i+j)] and
                               df["volume"].iloc[-(i+j)] > df["volume"].iloc[-(i+j+1)]
                               for j in range(3)):
                            merdiven_hacim = True
                            break
                    except:
                        continue

                merdiven_hacim_durumu = "✅ Var" if merdiven_hacim else "❌ Yok"

                # 4. Ortalama hacim oranı (Son 5 mum / 20 günlük ortalama)
                ort_hacim_orani = (df["volume"].iloc[-5:].mean() / df['volume_ma'].iloc[-1]) * 100 if df['volume_ma'].iloc[-1] > 0 else 0

                # 5. Son 5 mumda yeşil mum sayısı (Trend gücü için)
                son_5_yesil = sum(1 for i in range(1, 6) if df["close"].iloc[-i] > df["open"].iloc[-i])

                # ============================================================================
                # 🎯 KADEMELI SOL TEPE ANALİZİ (HH + LH)
                # ============================================================================
                # Kademeye göre lookback ayarla
                if veri_kademesi == "MİNİMUM":
                    hh_lookback = 30  # Daha kısa dönem
                    lh_lookback = 15
                else:
                    hh_lookback = 55  # Normal dönem
                    lh_lookback = 20

                if len(df) >= hh_lookback:
                    # 1. HH (Higher High) - 55 bar içinde en yüksek
                    son_55_hh = df['high'].iloc[-hh_lookback:].max()
                    hh_index = df['high'].iloc[-hh_lookback:].idxmax()
                    hh_bar_pozisyonu = len(df) - df.index.get_loc(hh_index)
                    hh_mesafesi = ((son_fiyat - son_55_hh) / son_55_hh) * 100

                    # 2. LH (Lower High) - Son 20 bar içinde yerel tepe
                    lh_value = 0
                    lh_bar_pozisyonu = 0
                    lh_mesafesi = 0

                    if hh_bar_pozisyonu > lh_lookback:  # HH yeterince eskiyse
                        son_20_high = df['high'].iloc[-lh_lookback:].max()
                        if son_20_high < son_55_hh:  # LH, HH'den düşükse
                            lh_value = son_20_high
                            lh_index = df['high'].iloc[-lh_lookback:].idxmax()
                            lh_bar_pozisyonu = len(df) - df.index.get_loc(lh_index)
                            lh_mesafesi = ((son_fiyat - lh_value) / lh_value) * 100

                    # 3. Hangisi daha yakın ve etkili?
                    if lh_value > 0 and lh_bar_pozisyonu < hh_bar_pozisyonu:
                        # LH daha yakın - onu kullan
                        direnç_degeri = lh_value
                        direnç_bar_pozisyonu = lh_bar_pozisyonu
                        direnç_mesafesi = lh_mesafesi
                        direnç_tipi = "LH"
                    else:
                        # HH kullan
                        direnç_degeri = son_55_hh
                        direnç_bar_pozisyonu = hh_bar_pozisyonu
                        direnç_mesafesi = hh_mesafesi
                        direnç_tipi = "HH"

                    # 4. Direnç durumunu belirle
                    fiyat_direncin_altinda = son_fiyat < direnç_degeri

                    if fiyat_direncin_altinda and direnç_mesafesi > -10:
                        sol_tepe_direnci = True
                        if direnç_mesafesi > -2:
                            sol_tepe_durumu = f"🔴 {direnç_tipi} Testinde"  # Çok yakın - KIRMIZI
                        else:
                            sol_tepe_durumu = f"🟠 {direnç_tipi} Direnci"   # Yakın - TURUNCU
                    elif fiyat_direncin_altinda:
                        sol_tepe_direnci = False
                        sol_tepe_durumu = f"🟢 {direnç_tipi} Uzak"         # Uzak - YEŞİL (İYİ!)
                    else:
                        sol_tepe_direnci = False
                        sol_tepe_durumu = f"🟢 {direnç_tipi} Geçildi"      # Geçildi - YEŞİL (ÇOK İYİ!)

                    # Tablo için değerler
                    hh_mesafesi = direnç_mesafesi
                    hh_bar_pozisyonu = direnç_bar_pozisyonu

                else:  # ← BU ELSE, "if len(df) >= hh_lookback:" BLOĞUNA AİT!
                    son_55_hh = son_fiyat
                    hh_mesafesi = 0
                    hh_bar_pozisyonu = 0
                    sol_tepe_durumu = "⚠️ Veri Yetersiz"
                    sol_tepe_direnci = False


                # ✅ YENİ: Dinamik EMA değerleri
                if tarama_secimi == '2':
                    ema1 = df[f'EMA{ema1_period}'].iloc[-1]
                    ema2 = df[f'EMA{ema2_period}'].iloc[-1]
                    ema3 = df[f'EMA{ema3_period}'].iloc[-1]
                    ema_siralama = "✅" if ema1 > ema2 > ema3 else "❌"
                    kucuk_emalar = [ema1, ema2, ema3]
                else:
                    ema13 = df['EMA13'].iloc[-1]
                    ema21 = df['EMA21'].iloc[-1]
                    ema34 = df['EMA34'].iloc[-1]
                    ema_siralama = "✅" if ema13 > ema21 > ema34 else "❌"
                    kucuk_emalar = [ema13, ema21, ema34]

                sikisma_orani_3ema = ((max(kucuk_emalar) - min(kucuk_emalar)) / min(kucuk_emalar)) * 100 if min(kucuk_emalar) > 0 else 0

                # ============================================================================
                # 🎯 KADEMELI EMA DİRENCİ KONTROLÜ
                # ============================================================================
                ema_direncler = {}

                # KADEME 1: Sadece EMA55 (Her zaman)
                if 'EMA55' in df.columns and df['EMA55'].iloc[-1] > son_fiyat:
                    ema_direncler['EMA55'] = df['EMA55'].iloc[-1]

                # KADEME 2: EMA89 ekle (Orta kademe)
                if ema89_hesaplandi:
                    if df['EMA89'].iloc[-1] > son_fiyat:
                        ema_direncler['EMA89'] = df['EMA89'].iloc[-1]

                # KADEME 3: EMA144 ekle (Tam kademe)
                if ema144_hesaplandi:
                    if df['EMA144'].iloc[-1] > son_fiyat:
                        ema_direncler['EMA144'] = df['EMA144'].iloc[-1]

                # Direnç durumu
                if ema_direncler:
                    yaklasan_ema_direnc = list(ema_direncler.keys())[0]
                    ema_direnc_durumu = yaklasan_ema_direnc
                elif veri_kademesi == "MİNİMUM":
                    ema_direnc_durumu = "⚠️ Kısmi Veri (EMA55)"
                else:
                    ema_direnc_durumu = "Önü Açık"


                # ============================================================================
                # 🎯 BİRLEŞİK DİRENÇ DURUMU (Öncelik: Sol Tepe > EMA > Önü Açık)
                # ============================================================================
                if sol_tepe_direnci:
                    birlesik_direnc = "Sol Tepe"
                elif ema_direnc_durumu != "Önü Açık" and "Veri Yetersiz" not in ema_direnc_durumu and "Kısmi Veri" not in ema_direnc_durumu:
                    birlesik_direnc = ema_direnc_durumu  # "EMA55", "EMA89", "EMA144"
                else:
                    birlesik_direnc = "Önü Açık"

                # ✅ YENİ: Uzaklık referansı dinamik
                if tarama_secimi == '1':
                    uzaklik_ref_ema_degeri = df[f'EMA{ref_ema_period}'].iloc[-1]
                elif tarama_secimi == '2':
                    uzaklik_ref_ema_degeri = df[f'EMA{ema1_period}'].iloc[-1]
                else:
                    # Sıkışma modu: Mevcut en büyük EMA'yı kullan
                    if ema89_hesaplandi:
                        uzaklik_ref_ema_degeri = df['EMA89'].iloc[-1]
                    else:
                        uzaklik_ref_ema_degeri = df['EMA55'].iloc[-1]

                uzaklik_yuzdesi = ((son_fiyat - uzaklik_ref_ema_degeri) / uzaklik_ref_ema_degeri) * 100
                if uzaklik_yuzdesi < 1.5: uzaklik_seviyesi = "Düşük"
                elif 1.5 <= uzaklik_yuzdesi < 4.0: uzaklik_seviyesi = "Orta"
                else: uzaklik_seviyesi = "Yüksek"

                # ============================================================================
                # 🎯 AKILLI TREND GÜCÜ HESAPLAMA
                # ============================================================================
                trend_ref_ema_period = ref_ema_period if tarama_secimi == '1' and f'EMA{ref_ema_period}' in df.columns else 55
                trend_ref_ema = df[f'EMA{trend_ref_ema_period}']
                ema_egimi = trend_ref_ema.iloc[-1] - trend_ref_ema.iloc[-5]
                ema_egim_yuzdesi = (ema_egimi / trend_ref_ema.iloc[-5]) * 100 if trend_ref_ema.iloc[-5] > 0 else 0

                # Sıkışma durumu kontrolü
                sikisik_mi = sikisma_orani_3ema < 2.0

                # AKILLI TREND SINIFLANDIRMASI (Sol Tepe Dahil)
                if ema_egim_yuzdesi > 0.5:  # EMA yukarı eğimli
                    if sol_tepe_direnci and son_mum_rengi == "Kırmızı":
                        trend_gucu = "⚠️ Yükseliş (Sol Tepe Direnci)"
                    elif sol_tepe_direnci and yesil_hacim_atak < 3:
                        trend_gucu = "⚠️ Konsolidasyon (HH Direnci)"
                    elif son_mum_rengi == "Yeşil" and son_5_yesil >= 4 and yesil_hacim_atak >= 6 and ort_hacim_orani > 120:
                        trend_gucu = "🔥 Çok Güçlü Yükseliş"
                    elif son_mum_rengi == "Yeşil" and son_5_yesil >= 3 and yesil_hacim_atak >= 4:
                        trend_gucu = "💪 Güçlü Yükseliş"
                    elif son_mum_rengi == "Yeşil" and son_10_yesil >= 5:
                        trend_gucu = "📈 Yükseliş"
                    elif sikisik_mi and son_10_yesil >= 4 and ort_hacim_orani < 100:
                        trend_gucu = "🌱 Yükseliş Başlangıcı"
                    elif son_10_yesil >= 4:
                        trend_gucu = "⚠️ Konsolidasyon"
                    else:
                        trend_gucu = "⚠️ Zayıf Yükseliş"

                elif ema_egim_yuzdesi < -0.5:  # EMA aşağı eğimli
                    trend_gucu = "📉 Düşüş"

                else:  # EMA yatay
                    if sikisik_mi and son_10_yesil >= 5:
                        trend_gucu = "🌱 Yükseliş Başlangıcı (Yatay)"
                    else:
                        trend_gucu = "➡️ Yatay"


                # ✅ YENİ: Sütun başlığı dinamik
                if tarama_secimi == '2':
                    ema_siralama_baslik = f"EMA Sıralaması {ema1_period}>{ema2_period}>{ema3_period}"
                elif tarama_secimi in ['3a', '3b', '3c']:
                    ema_siralama_text = '>'.join(map(str, sikisma_ema_periyotlari))
                    ema_siralama_baslik = f"EMA Sıralaması {ema_siralama_text}"
                else:
                    ema_siralama_baslik = "EMA Sıralaması 13>21>34"


                sonuc = {
                    "Hisse": hisse,
                    "Tarama Türü": "Kompozit (Hisse/XU100)" if kompozit_modu else "Normal Fiyat",
                    "Veri Kademesi": veri_kademesi,  # 🆕 YENİ
                    "Son Fiyat": son_fiyat,
                    "Son Mum": son_mum_rengi,
                    "Son 10 Yeşil Mum": son_10_yesil,
                    "Yeşil Hacim Atak": yesil_hacim_atak,
                    "Merdiven Hacim": merdiven_hacim_durumu,
                    "Hacim Gücü %": ort_hacim_orani,
                    ema_siralama_baslik: ema_siralama,
                    "Sıkışma %": sikisma_orani_3ema,

                    # 🆕 YENİ: İKİ AYRI DİRENÇ SÜTUNU
                    "EMA Direnci": ema_direnc_durumu,  # 🆕 YENİ
                    "Sol Tepe Durumu": sol_tepe_durumu,  # ✅ LH Direnci / HH Direnci
                    "Direnç Mesafesi %": hh_mesafesi,    # ✅ YENİ İSİM
                    "Direnç Bar Pozisyonu": hh_bar_pozisyonu,  # ✅ YENİ İSİM
                    "Direnç Durumu": birlesik_direnc,  # ✅ GÜNCELLENDİ
                    "Uzaklık Seviyesi": uzaklik_seviyesi,
                    "Fiyat-Ref. Uzaklık %": uzaklik_yuzdesi,  # ✅ KALACAK
                    "Trend Gücü": trend_gucu,  # ✅ YENİ AKILLI HESAPLAMA
                    "Hacim Atak (10 Mum)": hacim_atak_sayisi,
                    "Zaman Aralığı": interval_str
                }

                sonuclar.append(sonuc)

        except Exception as e:
            yetersiz.append(hisse); continue

# --- Sonuçları Gösterme ve Kaydetme ---
if sonuclar:
    df_son = pd.DataFrame(sonuclar)

    df_uzaklik_sirali = df_son.sort_values(by=['Fiyat-Ref. Uzaklık %', 'Sıkışma %', 'Hacim Atak (10 Mum)'], ascending=[True, True, False])
    df_sikisma_sirali = df_son.sort_values(by=['Sıkışma %', 'Fiyat-Ref. Uzaklık %', 'Hacim Atak (10 Mum)'], ascending=[True, True, False])

    print("\n✅ Tarama Tamamlandı. Sonuçlar Aşağıdadır:")
    print("\n" + "="*100 + "\n🎯 TABLO 1: RİSK ODAKLI SIRALAMA (Fiyata En Yakın Olanlar - Tam Liste)\n" + "="*100)
    display(format_tarama_sonucu(df_uzaklik_sirali))
    print("\n" + "="*100 + "\n💥 TABLO 2: PATLAMA POTANSİYELİ ODAKLI SIRALAMA (En Sıkışık Olanlar - Tam Liste)\n" + "="*100)
    display(format_tarama_sonucu(df_sikisma_sirali))

    if len(df_son) > 25:
        print("\n" + "="*100 + f"\n🏆 TOP 25 - RİSK ODAKLI LİSTE (Toplam {len(df_son)} sonuç arasından)\n" + "="*100)
        display(format_tarama_sonucu(df_uzaklik_sirali.head(25)))
        print("\n" + "="*100 + f"\n🏆 TOP 25 - PATLAMA POTANSİYELİ LİSTESİ (Toplam {len(df_son)} sonuç arasından)\n" + "="*100)
        display(format_tarama_sonucu(df_sikisma_sirali.head(25)))


    if tarama_secimi == '1':
        kullanilan_strateji = "EMA_DESTEK"
    elif tarama_secimi == '2':
        # ✅ YENİ: Dosya adına EMA periyotlarını ekle
        kullanilan_strateji = f"IDEAL_DIZILIM_{ema1_period}_{ema2_period}_{ema3_period}"
    elif tarama_secimi in ['3a', '3b', '3c']:
        ema_text = '_'.join(map(str, sikisma_ema_periyotlari))
        kullanilan_strateji = f"EMA_SIKISMA_{ema_sayisi}EMA_{ema_text}"
    else:
        kullanilan_strateji = "EMA_SIKISMA_5"


        # Kompozit bilgisini dosya adına ekle
    kompozit_ek = "_KOMPOZIT" if kompozit_modu else ""
    filename = f"PRO_TARAMA_{kullanilan_strateji}{kompozit_ek}_{interval_str.replace(' ', '')}_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df_uzaklik_sirali.to_excel(writer, sheet_name='Risk_Odakli_Siralama', index=False)
            df_sikisma_sirali.to_excel(writer, sheet_name='Sikisma_Odakli_Siralama', index=False)
        print(f"\n💾 Excel'e kaydedildi: {filename} (2 sayfa olarak)")
    except Exception as e:
        print(f"\n❌ Excel'e kaydederken hata oluştu: {e}")

    tarih_saat = datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"\n📋 {tarih_saat} tarihinde yapılan {kullanilan_strateji.replace('_', ' ')} Tarama Özeti:")
    print(f"📊 Tarama Türü: {'Kompozit (Hisse/XU100)' if kompozit_modu else 'Normal Fiyat'}")
    print(f"📊 Zaman Aralığı: {interval_str}")

    # Kompozit mod ise XU100 bilgisini ekle
    if kompozit_modu and xu100_son_fiyat:
        print(f"📈 Referans XU100 Değeri: {xu100_son_fiyat:,.2f}")
        print(f"ℹ️  NOT: Tablodaki 'Son Fiyat' değerleri, hissenin XU100'e oranını gösterir.")

    if tarama_secimi == '1':
        print(f"📈 Referans EMA Periyodu: {ref_ema_period}")
    elif tarama_secimi == '2':
        print(f"🎯 EMA Dizilimi: {ema1_period} > {ema2_period} > {ema3_period}")
    elif tarama_secimi in ['3a', '3b', '3c']:
        print(f"🔥 Seçilen EMA'lar: {sikisma_ema_periyotlari}")
        print(f"📉 Maksimum Sıkışma Oranı: %{maks_sikisma_orani}")
    print(f"✅ Toplam {len(df_son)} adet hisse bulundu.")
    if yetersiz: print(f"\n⚠️ Veri yetersizliği/hata nedeniyle taranamayan {len(yetersiz)} adet hisse: {', '.join(yetersiz)}")
else:
    print("\n⚠️ Belirtilen kriterlere uygun hisse bulunamadı.")
    if yetersiz: print(f"\n⚠️ Veri yetersizliği/hata nedeniyle taranamayan {len(yetersiz)} adet hisse: {', '.join(yetersiz)}")


# EMA1>EMA2>EMA3 KURALINA UYAN HİSSELER İÇİN TARAMA, SORUNSUZ ÇALIŞIYOR....!!!!! YANİ EMA PERİYOTLARINI SİZ SEÇERSİNİZ.



  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.5 MB/s eta 0:00:00

                    🚀 ÇOK STRATEJİLİ EMA TARAMA ROBOTU v3.2                     

💡 Bu araç, BIST hisselerini güçlü EMA stratejilerine göre tarar.
⚠️  Yatırım tavsiyesi değildir. Eğitim amaçlıdır.


                            📊 HİSSE FİLTRELEME SEÇİMİ                           

📈 TARAMA TÜRLERİ:

1️⃣  : Tüm BIST Hisseleri (Normal Fiyat)
    └─ Hisse fiyatları olduğu gibi kullanılır
    └─ Klasik teknik analiz
    └─ Tüm stratejiler için uygun

2️⃣  : Kompozit BIST Hisseleri (Hisse / XU100)
    └─ Hisse fiyatı XU100 endeksine bölünür
    └─ Göreceli güç analizi
    └─ Endeksten bağımsız performans ölçümü

💡 ÖNERİ: Kompozit mod, hissenin piyasaya göre gerçek gücünü gösterir

👉 Tarama türü seçin (1-2): 1
✅ Seçilen: Normal BIST Taraması


                                🎯 STRATEJİ SEÇİMİ                               

📈 MEVCUT STRATEJİLER:

1️⃣  : EMA Destek Stratejisi

✅ Kullanılacak: EMA 13 > 21 > 34


                              ⏳ 🚀 TARAMA BAŞLIYOR                               
📊 Strateji: İdeal Dizilim
⏰ Zaman Aralığı: Haftalık
🎯 EMA Dizilimi: 13 > 21 > 34

📊 KADEMELI MUM KONTROLÜ:
   ├─ Minimum (Taramaya Dahil): 75 mum (EMA55'e kadar)
   ├─ Orta (EMA89 + Direnç): 109 mum
   └─ Tam (EMA144 + Tam Direnç): 164 mum

ℹ️ Hesaplanacak EMA'lar: EMA13, EMA21, EMA34, EMA55, EMA89, EMA144
ℹ️ Normal mod seçildi, XU100 verisi çekilmeyecek.


📈 BIST Sembolleri Çekiliyor...
✅ 619 adet BIST sembolü çekildi.


📊 Hisseler Taranıyor: 100%|██████████| 619/619 [05:37<00:00,  1.83it/s]



✅ Tarama Tamamlandı. Sonuçlar Aşağıdadır:

🎯 TABLO 1: RİSK ODAKLI SIRALAMA (Fiyata En Yakın Olanlar - Tam Liste)


,Hisse,Tarama Türü,Veri Kademesi,Son Fiyat,Son Mum,Son 10 Yeşil Mum,Yeşil Hacim Atak,Merdiven Hacim,Hacim Gücü %,EMA Sıralaması 13>21>34,Sıkışma %,EMA Direnci,Sol Tepe Durumu,Direnç Mesafesi %,Direnç Bar Pozisyonu,Direnç Durumu,Uzaklık Seviyesi,Fiyat-Ref. Uzaklık %,Trend Gücü,Hacim Atak (10 Mum),Zaman Aralığı
204,UNLU,Normal Fiyat,TAM,15.350000,Kırmızı,5,1,❌ Yok,81%,✅,0.51%,Önü Açık,🟢 HH Uzak,-15.47%,18,Önü Açık,Düşük,0.01%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık
37,ECZYT,Normal Fiyat,TAM,305.750000,Kırmızı,3,1,❌ Yok,65%,✅,10.64%,Önü Açık,🟢 HH Uzak,-23.42%,9,Önü Açık,Düşük,0.07%,⚠️ Zayıf Yükseliş,1,Haftalık
190,OZGYO,Normal Fiyat,TAM,2.060000,Kırmızı,5,2,❌ Yok,97%,✅,5.45%,Önü Açık,🟢 HH Uzak,-18.74%,6,Önü Açık,Düşük,0.34%,⚠️ Konsolidasyon,2,Haftalık
177,DOCO,Normal Fiyat,TAM,10142.500000,Kırmızı,6,5,❌ Yok,94%,✅,5.74%,Önü Açık,🟢 HH Uzak,-12.94%,16,Önü Açık,Düşük,0.44%,⚠️ Konsolidasyon,5,Haftalık
120,TURSG,Normal Fiyat,TAM,11.720000,Yeşil,3,2,❌ Yok,110%,✅,9.18%,Önü Açık,🟢 HH Uzak,-17.29%,11,Önü Açık,Düşük,0.52%,⚠️ Zayıf Yükseliş,2,Haftalık
137,LUKSK,Normal Fiyat,TAM,111.200000,Kırmızı,2,1,❌ Yok,111%,✅,4.85%,Önü Açık,🟢 HH Uzak,-23.99%,18,Önü Açık,Düşük,0.68%,⚠️ Zayıf Yükseliş,1,Haftalık
135,DUNYH,Normal Fiyat,TAM,110.900000,Yeşil,5,1,❌ Yok,69%,✅,14.80%,Önü Açık,🟢 HH Uzak,-23.62%,16,Önü Açık,Düşük,0.68%,📈 Yükseliş,1,Haftalık
152,TGSAS,Normal Fiyat,TAM,171.700000,Yeşil,3,1,❌ Yok,129%,✅,8.85%,Önü Açık,🟢 LH Uzak,-32.00%,20,Önü Açık,Düşük,0.78%,⚠️ Zayıf Yükseliş,1,Haftalık
1,PCILT,Normal Fiyat,TAM,22.900000,Yeşil,4,0,❌ Yok,38%,✅,6.35%,Önü Açık,🟢 HH Uzak,-38.11%,18,Önü Açık,Düşük,0.79%,⚠️ Konsolidasyon,0,Haftalık
113,FMIZP,Normal Fiyat,TAM,327.000000,Kırmızı,5,1,❌ Yok,71%,✅,1.06%,Önü Açık,🟢 LH Uzak,-12.33%,12,Önü Açık,Düşük,0.81%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık


...


🏆 TOP 25 - RİSK ODAKLI LİSTE (Toplam 246 sonuç arasından)


,Hisse,Tarama Türü,Veri Kademesi,Son Fiyat,Son Mum,Son 10 Yeşil Mum,Yeşil Hacim Atak,Merdiven Hacim,Hacim Gücü %,EMA Sıralaması 13>21>34,Sıkışma %,EMA Direnci,Sol Tepe Durumu,Direnç Mesafesi %,Direnç Bar Pozisyonu,Direnç Durumu,Uzaklık Seviyesi,Fiyat-Ref. Uzaklık %,Trend Gücü,Hacim Atak (10 Mum),Zaman Aralığı
204,UNLU,Normal Fiyat,TAM,15.350000,Kırmızı,5,1,❌ Yok,81%,✅,0.51%,Önü Açık,🟢 HH Uzak,-15.47%,18,Önü Açık,Düşük,0.01%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık
37,ECZYT,Normal Fiyat,TAM,305.750000,Kırmızı,3,1,❌ Yok,65%,✅,10.64%,Önü Açık,🟢 HH Uzak,-23.42%,9,Önü Açık,Düşük,0.07%,⚠️ Zayıf Yükseliş,1,Haftalık
190,OZGYO,Normal Fiyat,TAM,2.060000,Kırmızı,5,2,❌ Yok,97%,✅,5.45%,Önü Açık,🟢 HH Uzak,-18.74%,6,Önü Açık,Düşük,0.34%,⚠️ Konsolidasyon,2,Haftalık
177,DOCO,Normal Fiyat,TAM,10142.500000,Kırmızı,6,5,❌ Yok,94%,✅,5.74%,Önü Açık,🟢 HH Uzak,-12.94%,16,Önü Açık,Düşük,0.44%,⚠️ Konsolidasyon,5,Haftalık
120,TURSG,Normal Fiyat,TAM,11.720000,Yeşil,3,2,❌ Yok,110%,✅,9.18%,Önü Açık,🟢 HH Uzak,-17.29%,11,Önü Açık,Düşük,0.52%,⚠️ Zayıf Yükseliş,2,Haftalık
137,LUKSK,Normal Fiyat,TAM,111.200000,Kırmızı,2,1,❌ Yok,111%,✅,4.85%,Önü Açık,🟢 HH Uzak,-23.99%,18,Önü Açık,Düşük,0.68%,⚠️ Zayıf Yükseliş,1,Haftalık
135,DUNYH,Normal Fiyat,TAM,110.900000,Yeşil,5,1,❌ Yok,69%,✅,14.80%,Önü Açık,🟢 HH Uzak,-23.62%,16,Önü Açık,Düşük,0.68%,📈 Yükseliş,1,Haftalık
152,TGSAS,Normal Fiyat,TAM,171.700000,Yeşil,3,1,❌ Yok,129%,✅,8.85%,Önü Açık,🟢 LH Uzak,-32.00%,20,Önü Açık,Düşük,0.78%,⚠️ Zayıf Yükseliş,1,Haftalık
1,PCILT,Normal Fiyat,TAM,22.900000,Yeşil,4,0,❌ Yok,38%,✅,6.35%,Önü Açık,🟢 HH Uzak,-38.11%,18,Önü Açık,Düşük,0.79%,⚠️ Konsolidasyon,0,Haftalık
113,FMIZP,Normal Fiyat,TAM,327.000000,Kırmızı,5,1,❌ Yok,71%,✅,1.06%,Önü Açık,🟢 LH Uzak,-12.33%,12,Önü Açık,Düşük,0.81%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık



🏆 TOP 25 - PATLAMA POTANSİYELİ LİSTESİ (Toplam 246 sonuç arasından)


,Hisse,Tarama Türü,Veri Kademesi,Son Fiyat,Son Mum,Son 10 Yeşil Mum,Yeşil Hacim Atak,Merdiven Hacim,Hacim Gücü %,EMA Sıralaması 13>21>34,Sıkışma %,EMA Direnci,Sol Tepe Durumu,Direnç Mesafesi %,Direnç Bar Pozisyonu,Direnç Durumu,Uzaklık Seviyesi,Fiyat-Ref. Uzaklık %,Trend Gücü,Hacim Atak (10 Mum),Zaman Aralığı
16,TATGD,Normal Fiyat,TAM,13.360000,Kırmızı,4,1,❌ Yok,78%,✅,0.13%,EMA89,🟠 LH Direnci,-8.87%,12,Sol Tepe,Yüksek,4.41%,➡️ Yatay,1,Haftalık
204,UNLU,Normal Fiyat,TAM,15.350000,Kırmızı,5,1,❌ Yok,81%,✅,0.51%,Önü Açık,🟢 HH Uzak,-15.47%,18,Önü Açık,Düşük,0.01%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık
144,CEMTS,Normal Fiyat,TAM,11.530000,Yeşil,6,0,❌ Yok,51%,✅,0.58%,Önü Açık,🟢 LH Uzak,-10.62%,16,Önü Açık,Orta,1.86%,🌱 Yükseliş Başlangıcı (Yatay),0,Haftalık
127,ULUFA,Normal Fiyat,TAM,3.970000,Yeşil,4,1,❌ Yok,54%,✅,0.60%,Önü Açık,🟢 LH Uzak,-13.70%,20,Önü Açık,Orta,3.21%,➡️ Yatay,1,Haftalık
69,AGYO,Normal Fiyat,TAM,7.790000,Yeşil,4,1,❌ Yok,86%,✅,0.80%,Önü Açık,🟠 LH Direnci,-5.12%,18,Sol Tepe,Yüksek,6.05%,➡️ Yatay,1,Haftalık
23,FROTO,Normal Fiyat,TAM,103.100000,Kırmızı,5,3,✅ Var,89%,✅,0.93%,Önü Açık,🟠 LH Direnci,-6.27%,18,Sol Tepe,Yüksek,4.81%,⚠️ Yükseliş (Sol Tepe Direnci),3,Haftalık
157,BIZIM,Normal Fiyat,TAM,29.920000,Yeşil,7,1,❌ Yok,82%,✅,0.95%,Önü Açık,🟠 HH Direnci,-3.48%,2,Sol Tepe,Yüksek,9.14%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık
89,INGRM,Normal Fiyat,TAM,487.750000,Yeşil,5,1,❌ Yok,71%,✅,1.03%,Önü Açık,🟢 HH Uzak,-21.46%,20,Önü Açık,Yüksek,14.13%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık
113,FMIZP,Normal Fiyat,TAM,327.000000,Kırmızı,5,1,❌ Yok,71%,✅,1.06%,Önü Açık,🟢 LH Uzak,-12.33%,12,Önü Açık,Düşük,0.81%,🌱 Yükseliş Başlangıcı (Yatay),1,Haftalık
13,MZHLD,Normal Fiyat,TAM,7.310000,Kırmızı,4,2,❌ Yok,88%,✅,1.09%,Önü Açık,🟢 HH Uzak,-14.00%,18,Önü Açık,Yüksek,10.70%,➡️ Yatay,2,Haftalık



💾 Excel'e kaydedildi: PRO_TARAMA_IDEAL_DIZILIM_13_21_34_Haftalık_20260121_2320.xlsx (2 sayfa olarak)

📋 2026-01-21 23:20 tarihinde yapılan IDEAL DIZILIM 13 21 34 Tarama Özeti:
📊 Tarama Türü: Normal Fiyat
📊 Zaman Aralığı: Haftalık
🎯 EMA Dizilimi: 13 > 21 > 34
✅ Toplam 246 adet hisse bulundu.

⚠️ Veri yetersizliği/hata nedeniyle taranamayan 56 adet hisse: VAKFA, OPTLR, KLYPV, ECOGR, MEYSU, FRMPL, EFOR, TCKRC, BIGEN, VSNMD, MOPAS, LILAK, KOCMT, OZATD, ISGLK, YIGIT, ZGYO, CGCAM, ZERGY, OPT25, GLRMK, BINBN, SMRVA, ALTNY, ARFYE, RGYAS, DCTTR, HRKET, QTEMZ, AHSGY, ENDAE, DURKN, AKGRT, DOKTA, BAHKM, OPTGY, HOROZ, GUNDG, DMLKT, OPK30, EGEGY, CEMZY, SEGMN, BULGS, KOTON, ALKLC, AKFIS, PAHOL, SERNT, ONRYT, MARMR, ARMGD, DSTKF, DOFRB, BALSU, OZYSR
